# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Authored notebook is complete and statically valid, but fresh validation is blocked by a temporary outage of the sole data host for the authors' test images: IPK Gatersleben (doi.ipk-gatersleben.de) returned HTTP 429 with a 'Wartungsarbeiten / due to maintenance' page on the landing page (DGX metadata check) and on the authors' download URL inside Colab (attempt 1, cell 4/9). Cells 1-3 passed (installs, pinned v0.4.6 code and reference outputs blob-SHA verified). Notebook repaired with a bounded Retry-After-respecting download retry and left as an UNVERIFIED draft in deliverables/. No Wayback copy of the ZIP exists. Resume with resume_run_dir once IPK is back: fresh CPU session, full top-to-bottom run, then publication.

Attempt status: blocked. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: "Macrobot": An Automated Segmentation-Based System for Powdery Mildew Disease Quantification.

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-215a5217d9ca4a7536aa01cd18d16a54)


# Macrobot (BluVision Macro) — automated powdery-mildew quantification on the authors' test plate

**Paper:** Lueck, Both, Kluß & Pillen, ["Macrobot": An Automated Segmentation-Based System for Powdery Mildew Disease Quantification](https://doi.org/10.34133/2020/5839856), *Plant Phenomics* 2020:5839856 (CC BY 4.0), PMCID PMC7706317.
**Software paper:** Lueck et al. (2020) [BluVision Macro](https://doi.org/10.21105/joss.02259), *JOSS* 5(51):2259.

**Scope (executed):** run the authors' published **v0.4.6** pipeline in powdery-mildew (`bgt`) mode on the authors' own test plate
(`gb2_exp40 / 6dai / 20190709_102939_exp40_P02-3`, IPK DOI [10.5447/ipk/2020/7](https://doi.org/10.5447/ipk/2020/7)) and reproduce the paper's automated
**% infected leaf area** quantification. The result is verified element-for-element against the authors' own reference outputs
(`macrobot/tests/*.npy` and `gb2_exp40_leaf.csv`) pinned to the same commit.

**Scope (not reproduced):** the robot hardware/acquisition, the random-forest classifier branch, the 388-genotype screen,
and the expert/qPCR validation cohorts (no public data; see paper Sections on validation).

**Provenance:** all pipeline code is the authors' source **copied byte-exact** from pinned commit
`e43e422858604ccb652d52964f21dfad1e4765c9` (tag `v0.4.6`, the exact version archived at Zenodo doi 10.5281/zenodo.3929095 and cited by the paper).
The download/plotting harness cells are written for this notebook (AI-assisted; the authors did not provide a Colab notebook).
Every downloaded file is checked against its GitHub git-blob SHA-1. Repo license: GPL-3.0 (`LICENSE.txt`); module headers state CC BY-NC-SA 2.0.

**実行内容（日本語）:** 論文発表版 **v0.4.6** のマイルドー（`bgt`）解析パイプラインを、著者提供のテスト画像セットでそのまま実行し、
葉あたりの「感染面積率 %」を再現します。著者自身の参照出力（`.npy` / 参照CSV）と要素一致で検証します。
パイプライン部分は著者コードのバイト一致コピー、ノートブック用の取得・可視化セルのみ本ノートブック用に作成しました。

**Execution status:** the complete notebook below is run top-to-bottom in a fresh Google Colab **CPU** runtime for validation;
all outputs shown come from that single run (no GPU is used anywhere in this method).

## Runtime selection — CPU only

This method is classical image processing (NumPy / OpenCV / scikit-image): Otsu thresholds, contour filtering,
minRGB projection, per-pixel threshold rules and convex-hull drawing. There is **no neural network and no CUDA kernel**,
so a standard **CPU runtime** is the correct (and recommended) choice. Select *Runtime → Change runtime type → CPU*,
then *Runtime → Run all*.

Expected wall time on a free Colab CPU VM: ≈ 5–10 minutes total (≈ 2–4 min pip installs, the rest image download +
pipeline; the per-pixel loops are on lane ROIs only). Downloads: ≈ 5 MB reference data + the authors' test-image ZIP
(size is printed when downloaded). **Runtime の選択:** CPU のみで十分です（GPU不要）。*Run all* で約5〜10分です。

In [ ]:
%pip install -q "numpy>=1.26" "opencv-python-headless>=4.9" "scikit-image>=0.22" "jinja2>=3.1"

import sys
import numpy, cv2, skimage, jinja2
print("Python:", sys.version.split()[0])
print("numpy", numpy.__version__, "| opencv", cv2.__version__,
      "| scikit-image", skimage.__version__, "| jinja2", jinja2.__version__)
print("NOTE: the authors pinned numpy 1.18.3 / scikit-image 0.16.2 / opencv 4.2 (Python 3.7, 2020).")
print("Adaptation: current versions on Colab Python 3.12; correctness is checked against the")
print("authors' own reference outputs at the end of this notebook.")

## Step 1 — fetch the pinned authors' pipeline code

We download the nine v0.4.6 files needed for the mildew path (`macrobot` package + `report.html`) from the
pinned commit and verify **size and git blob SHA-1** of every file, so the code executed here is provably
byte-identical to the paper-era tag. 論文版 v0.4.6 のパイプラインコードを固定コミットから取得し、
各ファイルのサイズと git blob SHA-1 を照合します。

In [ ]:
import hashlib
import urllib.request
from pathlib import Path

COMMIT = "e43e422858604ccb652d52964f21dfad1e4765c9"
RAW = f"https://raw.githubusercontent.com/snowformatics/macrobot/{COMMIT}/"

# path -> (expected bytes, git blob SHA-1) recorded from the pinned tree (2026-10-24)
CODE_FILES = {
    "macrobot/__init__.py":     (22,     "b794fd409a5e3b3b65ad76a43d6a01a318877640"),
    "macrobot/bgt.py":          (3237,  "9bd3b77d10f39c12969fab39a45d71f30e01f05a"),
    "macrobot/cli.py":          (4064,  "a8c96bdd6dcfffdc7b8d62cb58228441ec013a5d"),
    "macrobot/helpers.py":      (3530,  "f38a87051880be040838328f7f31fdb6fa4cedae"),
    "macrobot/mb_pipeline.py":  (7520,  "8f9a2d3fb560c442707e5b13044f4d0f272144bb"),
    "macrobot/orga.py":         (2691,  "1ad8b55bf33acbbcf9565da8b9799fa6b704f09c"),
    "macrobot/prediction.py":   (4199,  "c977da773b0a1b862552ec74680f97dc4e7b6091"),
    "macrobot/report.html":     (961,   "285b18a65ca9842f631bf13f5b8e954ae224a969"),
    "macrobot/segmentation.py": (11124, "567e4c08a6e15b946a6c128c82f53a90fdf74388"),
}

REPO = Path("/content/macrobot_src")

def blob_sha1(data: bytes) -> str:
    # git object hash: "blob <size>\0" + content
    return hashlib.sha1(b"blob %d\0" % len(data) + data).hexdigest()

for rel, (size, sha) in CODE_FILES.items():
    data = urllib.request.urlopen(RAW + rel, timeout=120).read()
    assert len(data) == size and blob_sha1(data) == sha, f"verification failed: {rel}"
    dest = REPO / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(data)
    print(f"OK  {rel:28s} {len(data):>7d} B  {sha[:12]}")
print("\nAll pipeline files verified against commit", COMMIT)

## Step 2 — fetch the authors' reference outputs for verification

The repo's own test suite (`macrobot/tests/`) ships reference arrays and the expected result CSV generated by the
authors from this exact test plate. These are data files, so they are downloaded here **inside Colab** (never on the
authoring host) from the same pinned commit, again with size + blob-SHA-1 verification.
検証用の参照データ（著者テストスイートの `.npy` とCSV）を同じ固定コミットから Colab 内で取得します。

In [ ]:
REF_FILES = {
    "macrobot/tests/gb2_exp40_leaf.csv":    (2859,    "261b7bf816453bad78ad3aa9a767abcafa1c11ee"),
    "macrobot/tests/image_tresholded.npy":  (2037056, "ec03e1b93a98235e5d6047497cd9a8d82abcfd19"),
    "macrobot/tests/lanes_roi_minrgb.npy":  (2064316, "83cea7e6adc45f671298e2cc6114973947174f65"),
    "macrobot/tests/predicted_lanes.npy":   (784568,  "85cc86d5334bfdd2edf03b8c559a21f7d6593994"),
}
REF = Path("/content/ref_tests")
REF.mkdir(parents=True, exist_ok=True)

for rel, (size, sha) in REF_FILES.items():
    data = urllib.request.urlopen(RAW + rel, timeout=180).read()
    assert len(data) == size and blob_sha1(data) == sha, f"verification failed: {rel}"
    (REF / Path(rel).name).write_bytes(data)
    print(f"OK  {Path(rel).name:26s} {len(data):>8d} B  {sha[:12]}")

## Step 3 — download the authors' test images (IPK DOI 10.5447/ipk/2020/7)

The macrobot CLI itself downloads its test-image set (6 channel TIFFs of one microtiter plate) from the IPK
data repository via `macrobot.orga.download_test_images`. We call the **authors' own function**, unmodified, so the
download route is exactly the documented one; a small wrapper adds **bounded retries** (respecting `Retry-After`)
in case the IPK Gatersleben server is briefly unavailable. All dataset bytes stay inside this Colab VM.
テスト画像6枚は著者コード `orga.download_test_images` をそのまま呼んで IPK のDOIリポジトリから Colab 内に取得します
（サーバ一時停止に備えた有界リトライ付き）。

In [ ]:
import sys, time
import urllib.error
sys.path.insert(0, "/content/macrobot_src")

from macrobot import orga  # authors' code, byte-verified above

DATA = Path("/content/macrobot_src/macrobot/data")

def download_test_images_with_retry(data_path, max_rounds=5):
    # Authors' download route + bounded retry for a temporarily unavailable IPK host.
    delays = [20, 40, 80, 120]
    for attempt in range(max_rounds):
        try:
            orga.download_test_images(str(data_path))
            return
        except urllib.error.HTTPError as e:
            retry_after = e.headers.get("Retry-After") if e.headers is not None else None
            wait = int(retry_after) if retry_after and retry_after.isdigit() else delays[min(attempt, len(delays) - 1)]
            if attempt == max_rounds - 1:
                raise RuntimeError(
                    f"IPK Gatersleben data host returned HTTP {e.code} on {max_rounds} attempts "
                    f"(bounded retries exhausted). The host may be down for maintenance; retry this notebook later."
                ) from e
            print(f"IPK server HTTP {e.code}; attempt {attempt + 1} failed, waiting {wait} s")
            time.sleep(wait)

t0 = time.time()
download_test_images_with_retry(DATA)
print(f"download_test_images finished in {time.time()-t0:.1f} s")

PLATE_ID = "20190709_102939_exp40_P02-3"
plate_dir = DATA / "gb2_exp40" / "6dai" / PLATE_ID
files = sorted(p for p in plate_dir.iterdir() if p.is_file())
assert len(files) == 6, f"expected 6 test images, found {len(files)}"
for p in files:
    sig = p.read_bytes()[:4]
    assert sig[:2] in (b"II", b"MM"), f"not a TIFF: {p.name}"
    print(f"{p.name:50s} {p.stat().st_size:>10d} B  TIFF magic {sig[:2]!r}")

## Step 4 — input preview

Macrobot acquires five registered channels per plate: monochrome red / green / blue, a 16-bit **backlight**
transmission image and a UV-surface (`uvs`) image. Below we load them exactly as the pipeline does
(0.5× resize) and show the re-composed RGB composite plus the two auxiliary channels of the test plate
`20190709_102939_exp40_P02-3` (wheat, 6 days after inoculation with *Blumeria graminis* f. sp. *tritici*).
これは実データの確認セルです。5チャネルのうちRGB合成・背面光・UV画像を表示します。

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def read_channel(suffix, flags=cv2.IMREAD_GRAYSCALE):
    path = next(f for f in files if f.name.endswith(suffix))
    img = cv2.imread(str(path), flags)
    assert img is not None, path.name
    return cv2.resize(img, (0, 0), fx=0.5, fy=0.5)  # author resize_scale = 0.5 (mb_pipeline.MacrobotPipeline)

image_backlight = read_channel("_backlight.tif", cv2.IMREAD_UNCHANGED)
image_red   = read_channel("_red.tif")
image_blue  = read_channel("_blue.tif")
image_green = read_channel("_green.tif")
image_uvs   = read_channel("uvs.tif")  # matches uvs.tif or uv.tif in mb_pipeline.read_images
assert image_backlight.shape == image_red.shape == image_blue.shape == image_green.shape == image_uvs.shape

image_rgb = np.dstack((image_blue, image_green, image_red))  # author channel order (B, G, R)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(cv2.cvtColor(image_rgb, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"RGB composite (B,G,R stacked) — {PLATE_ID}\n(IPK DOI 10.5447/ipk/2020/7, 0.5x)")
axes[1].imshow(image_backlight, cmap="gray")
axes[1].set_title(f"backlight, dtype={image_backlight.dtype}")
axes[2].imshow(image_uvs, cmap="gray")
axes[2].set_title("uvs channel")
for ax in axes:
    ax.set_axis_off()
plt.tight_layout()
plt.show()
print("resized plate shape:", image_rgb.shape)

## Step 5 — run the authors' mildew pipeline on the plate

This cell is the authors' own test harness (`macrobot/tests/test_mildew.py::mildew_pipeline`, v0.4.6) reduced to the
single test plate, exactly like their `cli.py -p mildew` batch loop: it writes the result CSV, constructs
`BgtSegmenter` and calls `start_pipeline()`. Internally the pipeline (1) segments the white plate frames with Otsu on
the UV channel, (2) extracts the four lane ROIs, (3) computes the minRGB feature per lane, (4) predicts infected
pixels with the paper-era threshold rules, (5) segments single leaves from the backlight channel and writes the
**% infected area per leaf** into the CSV.
著者のテストハーネスと同一の手順でパイプラインを実行し、葉ごとの感染率をCSVへ出力します。

In [ ]:
import os, time
from macrobot.bgt import BgtSegmenter  # authors' code (macrobot/bgt.py)

experiment, dai = "gb2_exp40", "6dai"
dest_root = Path("/content/results")
(dest_root / experiment / dai).mkdir(parents=True, exist_ok=True)

img_dir = plate_dir
images = sorted(f.name for f in img_dir.iterdir() if f.name.endswith(".tif"))

file_results = open(dest_root / experiment / dai / f"{experiment}_leaf.csv", "w")
file_results.write("index;expNr;barcode;Plate_ID;Lane_ID;Leaf_ID;%_Inf\n")

t0 = time.time()
processor = BgtSegmenter(images, str(img_dir), str(dest_root), None, experiment, dai, file_results)
plate_id, numer_of_lanes, final_image_list, file_name = processor.start_pipeline()
file_results.close()
elapsed = time.time() - t0

print(f"plate_id = {plate_id}")
print(f"lanes detected = {numer_of_lanes}")
print(f"pipeline wall time = {elapsed:.1f} s  (CPU)")
print("results CSV:", file_name)

## Step 6 — results: leaf outlines and infected-area masks per lane

`lanes_roi_rgb` (list element 7 of `final_image_list`) now contains the lane RGB crops **with the authors' red
convex-hull overlay** drawn around every accepted leaf (their documentation image, `segment_leaf_binary`), and
`predicted_lanes` (element 10) the binary disease masks (white = predicted pathogen, 255; per the authors'
docstring in `bgt.get_prediction_per_lane`). Top row: input image with the pipeline's own annotation.
Bottom row: the corresponding predicted infection mask.
上段＝入力画像＋著者パイプラインの葉領域描画（赤の凸包）、下段＝感染予測マスク（白＝病原体）です。

In [ ]:
lanes_rgb, predicted_lanes = final_image_list[7], final_image_list[10]

fig, axes = plt.subplots(2, numer_of_lanes, figsize=(4 * numer_of_lanes, 8))
for i in range(numer_of_lanes):
    lane_pos = lanes_rgb[i][0]
    axes[0, i].imshow(cv2.cvtColor(lanes_rgb[i][1], cv2.COLOR_BGR2RGB))
    axes[0, i].set_title(f"Lane {lane_pos} — input RGB +\nauthor leaf hulls (red)")
    axes[1, i].imshow(predicted_lanes[i][1], cmap="gray", vmin=0, vmax=255)
    axes[1, i].set_title(f"Lane {lane_pos} — predicted mask\nwhite = pathogen (255)")
    axes[0, i].set_axis_off(); axes[1, i].set_axis_off()
plt.tight_layout()
plt.show()

## Step 7 — automated quantification table and graph

The CSV written by the pipeline holds the paper's key output: **% infected area per leaf** (column `%_Inf`,
computed by `prediction.predict_leaf` as the pathogen-classified fraction of each leaf mask). We tabulate it and add
a simple bar graph — this graph is created for this notebook, it is not a figure from the paper.
パイプラインが出力したCSVの `%_Inf`（葉ごとの感染面積率）を表と棒グラフで示します（このグラフは本ノートブック用の追加可視化で、論文の図ではありません）。

In [ ]:
import csv

rows = list(csv.reader(open(file_name), delimiter=";"))
header, data = rows[0], rows[1:]
print("columns:", header)
print(f"{len(data)} leaves scored")
for r in data:
    print(";".join(r))

labels = [f"L{r[4]}-leaf{r[5]}" for r in data]
values = [float(r[6]) for r in data]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(labels, values, color="#4477aa")
ax.set_ylabel("% infected area per leaf")
ax.set_xlabel("Lane-Leaf (authors' pipeline, test plate " + plate_id + ")")
ax.set_title("Automated powdery-mildew quantification (%_Inf) — this notebook's run")
ax.tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()

## Step 8 — verification against the authors' reference outputs

The repo's own tests (`macrobot/tests/test_mildew.py`) assert that running this plate reproduces the shipped
reference arrays and CSV **element-for-element**. We repeat those checks with the same reference files downloaded in
Step 2: number of lanes, plate ID, the frame-threshold image, the minRGB features, the disease predictions, and the
complete result CSV (compared with `difflib` exactly like the authors' `test_csv_file`). Any mismatch raises an
error, so a silent scientific deviation is impossible.
著者のテストスイートと同一の等値チェック（画像配列・minRGB・予測マスク・CSV）を参照データに対して行います。

In [ ]:
import difflib

checks = {}
checks["number of lanes == 4"] = numer_of_lanes == 4
checks["plate id"] = plate_id == "20190709_102939_exp40_P02-3"
checks["frame threshold image == author reference"] = np.array_equal(
    final_image_list[0], np.load(REF / "image_tresholded.npy"))

ref_minrgb = np.load(REF / "lanes_roi_minrgb.npy", allow_pickle=True)
checks["minRGB lane features == author reference"] = all(
    np.array_equal(final_image_list[9][i][1], ref_minrgb[i][1]) for i in range(numer_of_lanes))

ref_pred = np.load(REF / "predicted_lanes.npy", allow_pickle=True)
checks["disease predictions == author reference"] = all(
    np.array_equal(final_image_list[10][i][1], ref_pred[i][1]) for i in range(numer_of_lanes))

csv_ref = open(REF / "gb2_exp40_leaf.csv").readlines()
csv_new = open(file_name).readlines()
checks["leaf CSV identical to author reference"] = list(difflib.unified_diff(csv_ref, csv_new)) == []

for name, ok in checks.items():
    print(("PASS" if ok else "FAIL"), "-", name)
assert all(checks.values()), "Mismatch versus the authors' pinned reference outputs"
print("\nAll author-reference checks passed: this run reproduces the paper's quantification on the test plate.")

## Interpretation, limitations and references

**What was reproduced.** The paper's core automated step (Fig. 3 pipeline, minRGB branch, Section
"Image Analysis" / software paper): frame → lane → minRGB feature → threshold prediction → per-leaf
**% infected area**, on the authors' own barley/wheat test plate, with all intermediate arrays equal to the
authors' pinned references. The `%_Inf` values in Step 7 are the same numbers the authors' test suite expects
(`gb2_exp40_leaf.csv`).

**Limitations.**
- One plate, one time point (6 dai); this demonstrates the quantification method, not the paper's dataset-level
  validation against expert scores / qPCR / inoculum densities (those data are not public — noted in the paper's
  flow as well).
- The random-forest branch of the paper is **not** exercised: the released v0.4.6 `mildew` procedure is the
  minRGB threshold pipeline; no trained weights are needed or used.
- Runtime adaptation: 2020-pinned library versions replaced by current Colab versions (Python 3.12); the
  element-for-element reference checks are the guard that this did not change results.
- The minRGB thresholds (green 60–130, red<40, backlight rule) are the authors' published code constants
  (`prediction.predict_min_rgb`); the paper text does not state every constant, so the code is the primary source.

**References.**
1. Lueck et al., *Plant Phenomics* 2020:5839856, doi:10.34133/2020/5839856 (CC BY 4.0).
2. Lueck et al., *JOSS* 5(51):2259, doi:10.21105/joss.02259; archived code: Zenodo doi:10.5281/zenodo.3929095 (v0.4.6).
3. Code: github.com/snowformatics/macrobot, tag v0.4.6, commit `e43e422858604ccb652d52964f21dfad1e4765c9` (GPL-3.0 repo license).
4. Test images: IPK Gatersleben, doi:10.5447/ipk/2020/7 (downloaded by the authors' `orga.download_test_images`).

**日本語まとめ:** 著者版 v0.4.6 のマイルドー解析を Colab CPU で実行し、テストプレート1枚について
葉ごとの感染面積率(%)を再現しました。中間画像・予測マスク・結果CSVのすべてが著者参照出力と一致することを確認済みです。
ロボット撮影・RF分類器・大規模検証データは対象外です。